In [ ]:
import os
import sys

import numpy as np
import matplotlib.pyplot as plt
import torch
from PIL import Image

# The notebook lives in Examples/; make the repository root importable
sys.path.append(os.path.abspath('..'))
from hanon.data import to_tensor

# Use the GPU when available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("GPU available: ", torch.cuda.is_available())

In [ ]:
# Path to the image to restore (replace with your own jpg)
image_path = '../Data/Balzac.jpg'

# Load as grayscale, values in 0-255
img = np.array(Image.open(image_path).convert('L'), dtype=np.uint8)

print(f"Image shape: {img.shape}")
print(f"Value range: {img.min()} to {img.max()}")

In [ ]:
# Load the trained models
cnn_noise = torch.load('../CNN/dncnn_model.pth', weights_only=False)
cnn_blur = torch.load('../CNN/Blur_cnn_model.pth', weights_only=False)
cnn_noise.eval()
cnn_blur.eval()

with torch.no_grad():
    # 1) Denoise: predict the noise and subtract it from the input
    noisy_input = to_tensor(img).unsqueeze(0).to(device)
    noise_pred = cnn_noise(noisy_input)
    denoised = torch.clamp(noisy_input - noise_pred, 0, 1)

    # 2) Deblur the denoised image the same way. The deblurring model also
    # expects inputs in [0, 1], so the denoised tensor is fed in directly.
    blur_pred = cnn_blur(denoised)
    final_image = torch.clamp(denoised - blur_pred, 0, 1)

denoised = denoised.cpu().squeeze()
final_image = final_image.cpu().squeeze()

# Visualize: original, denoised, denoised + deblurred
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(img, cmap='gray')
plt.title("Original Image")
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(denoised, cmap='gray')
plt.title("Denoised Image")
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(final_image, cmap='gray')
plt.title("Deblurred Image")
plt.axis('off')

plt.tight_layout()
plt.show()